In [1]:
import pandas as pd
from pathlib import Path

In [2]:
RESULTS_DIR = Path("tournament_results_probe")  # change if needed

dfs = []
for f in RESULTS_DIR.glob("*.csv"):
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} files, {len(combined)} total games")
combined.head()

Loaded 6 files, 1200 total games


,opening_idx,opening_fen,white_policy,black_policy,discrete,result,termination,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total,source_file
0,3,rnbqk2r/ppp2ppp/5n2/2b5/2N1p3/3P4/PPP2PPP/RNBQ...,TokenBucket,FixedUniform,False,1/2-1/2,draw_rule,84,8949295,3432976,1050705,6567024,TokenBucket_vs_FixedUniform_continuous.csv
1,3,rnbqk2r/ppp2ppp/5n2/2b5/2N1p3/3P4/PPP2PPP/RNBQ...,FixedUniform,TokenBucket,False,1/2-1/2,draw_rule,83,3432756,8974269,6567244,1025731,TokenBucket_vs_FixedUniform_continuous.csv
2,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,TokenBucket,FixedUniform,False,1-0,budget_forfeit,129,7848032,0,2151968,10000222,TokenBucket_vs_FixedUniform_continuous.csv
3,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,FixedUniform,TokenBucket,False,1-0,resign_adjudication,120,618002,551925,9381998,9448075,TokenBucket_vs_FixedUniform_continuous.csv
4,1,rnb1k2r/pp2qppp/3bp3/3n4/2BP4/5N2/PP1B1PPP/RN1...,TokenBucket,FixedUniform,False,1/2-1/2,draw_rule,87,8448535,3276818,1551465,6723182,TokenBucket_vs_FixedUniform_continuous.csv


In [3]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   opening_idx             1200 non-null   int64 
 1   opening_fen             1200 non-null   object
 2   white_policy            1200 non-null   object
 3   black_policy            1200 non-null   object
 4   discrete                1200 non-null   bool  
 5   result                  1200 non-null   object
 6   termination             1200 non-null   object
 7   total_plies             1200 non-null   int64 
 8   white_budget_remaining  1200 non-null   int64 
 9   black_budget_remaining  1200 non-null   int64 
 10  white_nodes_total       1200 non-null   int64 
 11  black_nodes_total       1200 non-null   int64 
 12  source_file             1200 non-null   object
dtypes: bool(1), int64(6), object(6)
memory usage: 113.8+ KB


In [4]:
combined.describe()

,opening_idx,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total
count,1200.000000,1200.000000,1.200000e+03,1.200000e+03,1.200000e+03,1.200000e+03
mean,49.500000,108.911667,3.606518e+06,3.810788e+06,6.393496e+06,6.189224e+06
std,28.878105,27.038579,3.217126e+06,3.155900e+06,3.217141e+06,3.155915e+06
min,0.000000,17.000000,0.000000e+00,0.000000e+00,2.252410e+05,3.002510e+05
25%,24.750000,90.000000,2.576738e+05,5.670248e+05,3.428018e+06,3.176596e+06
50%,49.500000,112.000000,2.976143e+06,3.274088e+06,7.023857e+06,6.725912e+06
75%,74.250000,128.000000,6.571982e+06,6.823404e+06,9.742326e+06,9.432975e+06
max,99.000000,207.000000,9.774759e+06,9.699749e+06,1.000053e+07,1.000122e+07


In [5]:
combined["result"].value_counts()

result
1/2-1/2    485
0-1        379
1-0        336
Name: count, dtype: int64

In [6]:
combined["termination"].value_counts()

termination
budget_forfeit         356
draw_adjudication      347
resign_adjudication    181
checkmate              178
draw_rule              138
Name: count, dtype: int64

In [7]:
def tokenbucket_score(df):
    wins = draws = losses = 0
    
    for _, row in df.iterrows():
        if row["result"] == "1/2-1/2":
            draws += 1
        elif row["result"] == "1-0":
            if row["white_policy"] == "TokenBucket":
                wins += 1
            elif row["black_policy"] == "TokenBucket":
                losses += 1
        elif row["result"] == "0-1":
            if row["black_policy"] == "TokenBucket":
                wins += 1
            elif row["white_policy"] == "TokenBucket":
                losses += 1
    
    total = len(df)
    score = wins + 0.5 * draws
    return wins, draws, losses, 100 * score / total

tokenbucket_score(combined) # WDL + %

(239, 485, 476, 40.125)

In [8]:
combined.groupby("discrete").size()

for mode, df in combined.groupby("discrete"):
    print("\nDiscrete =", mode)
    print(tokenbucket_score(df))


Discrete = False
(151, 215, 234, 43.083333333333336)

Discrete = True
(88, 270, 242, 37.166666666666664)


In [9]:
combined["termination"].value_counts(normalize=True)

termination
budget_forfeit         0.296667
draw_adjudication      0.289167
resign_adjudication    0.150833
checkmate              0.148333
draw_rule              0.115000
Name: proportion, dtype: float64

In [10]:
combined[combined["termination"] == "budget_forfeit"].head()

,opening_idx,opening_fen,white_policy,black_policy,discrete,result,termination,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total,source_file
2,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,TokenBucket,FixedUniform,False,1-0,budget_forfeit,129,7848032,0,2151968,10000222,TokenBucket_vs_FixedUniform_continuous.csv
5,1,rnb1k2r/pp2qppp/3bp3/3n4/2BP4/5N2/PP1B1PPP/RN1...,FixedUniform,TokenBucket,False,0-1,budget_forfeit,128,0,1620957,10000118,8379043,TokenBucket_vs_FixedUniform_continuous.csv
11,5,rnbq1rk1/pp2bpp1/4p2p/2Pn4/8/2N1P1P1/PP1B1P1P/...,FixedUniform,TokenBucket,False,0-1,budget_forfeit,128,0,3222261,10000073,6777739,TokenBucket_vs_FixedUniform_continuous.csv
13,6,rnb1k2r/1pqpbppp/5n2/p3p3/2PP4/P7/NPQ2PPP/R1B1...,FixedUniform,TokenBucket,False,0-1,budget_forfeit,128,0,5997327,10000104,4002673,TokenBucket_vs_FixedUniform_continuous.csv
15,4,r1bqk2r/ppp2pp1/1bnp3p/3np3/2B1P3/5N1P/PPPPQPP...,FixedUniform,TokenBucket,False,0-1,budget_forfeit,128,0,115524,10000004,9884476,TokenBucket_vs_FixedUniform_continuous.csv


In [11]:
combined[["white_nodes_total", "black_nodes_total"]].describe()

,white_nodes_total,black_nodes_total
count,1.200000e+03,1.200000e+03
mean,6.393496e+06,6.189224e+06
std,3.217141e+06,3.155915e+06
min,2.252410e+05,3.002510e+05
25%,3.428018e+06,3.176596e+06
50%,7.023857e+06,6.725912e+06
75%,9.742326e+06,9.432975e+06
max,1.000053e+07,1.000122e+07


In [12]:
def extract_nodes(row):
    if row["white_policy"] == "TokenBucket":
        return row["white_nodes_total"], row["black_nodes_total"]
    else:
        return row["black_nodes_total"], row["white_nodes_total"]

tmp = combined.apply(lambda r: extract_nodes(r), axis=1, result_type="expand")
tmp.columns = ["tb_nodes", "opp_nodes"]

tmp.describe()

,tb_nodes,opp_nodes
count,1.200000e+03,1.200000e+03
mean,5.184489e+06,7.398231e+06
std,3.535022e+06,2.319899e+06
min,2.252410e+05,8.004940e+05
25%,1.676308e+06,5.368632e+06
50%,4.452518e+06,7.506007e+06
75%,8.929125e+06,9.935084e+06
max,1.000122e+07,1.000038e+07


In [13]:
def classify(row):
    if row["result"] == "1/2-1/2":
        return "draw"
    if row["result"] == "1-0":
        return "win" if row["white_policy"] == "TokenBucket" else "loss"
    if row["result"] == "0-1":
        return "win" if row["black_policy"] == "TokenBucket" else "loss"

combined["tb_result"] = combined.apply(classify, axis=1)

combined["tb_result"].value_counts()
pd.crosstab(combined["tb_result"], combined["termination"])

termination,budget_forfeit,checkmate,draw_adjudication,draw_rule,resign_adjudication
tb_result,,,,,
draw,0,0,347,138,0
loss,133,171,0,0,172
win,223,7,0,0,9


In [14]:
for name, df in combined.groupby("source_file"):
    print("\n", name)
    print(tokenbucket_score(df))


 TokenBucket_vs_FixedUniform_continuous.csv
(57, 74, 69, 47.0)

 TokenBucket_vs_FixedUniform_discrete.csv
(12, 100, 88, 31.0)

 TokenBucket_vs_Hyatt_continuous.csv
(43, 76, 81, 40.5)

 TokenBucket_vs_Hyatt_discrete.csv
(38, 85, 77, 40.25)

 TokenBucket_vs_SolakVuckovic_continuous.csv
(51, 65, 84, 41.75)

 TokenBucket_vs_SolakVuckovic_discrete.csv
(38, 85, 77, 40.25)


In [15]:
pd.crosstab(combined["tb_result"], combined["termination"])

termination,budget_forfeit,checkmate,draw_adjudication,draw_rule,resign_adjudication
tb_result,,,,,
draw,0,0,347,138,0
loss,133,171,0,0,172
win,223,7,0,0,9


In [16]:
tmp.groupby(combined["tb_result"]).mean()

,tb_nodes,opp_nodes
tb_result,,
draw,1.821678e+06,6.006827e+06
loss,8.090712e+06,7.616534e+06
win,6.220479e+06,9.787014e+06


In [17]:
combined.groupby(["discrete", "tb_result"]).size()

discrete  tb_result
False     draw         215
          loss         234
          win          151
True      draw         270
          loss         242
          win           88
dtype: int64